# Day 4 · Qwen2.5-VL 架构精读

**配套讲义**: [`days/day-04.md`](../days/day-04.md) ｜ **需要 GPU（云机器）**

复现 Qwen2.5-VL 的「按原图比例切 patch、算 visual token 数」逻辑，**算出的数字必须等于官方 processor 的 `<|image_pad|>` 数量**。这一条通过，说明你真的搞懂了它的分辨率处理。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w1.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 先猜，再算

**1024×768 的图，会产生多少个 visual token？** 把猜测写下来。

In [ ]:
guesses = {"448×448": None, "1024×1024": None, "1024×768": None}
print("把你猜的数字填进上面的字典，然后往下跑对比。\n")

import math
PATCH, MERGE = 14, 2
for size in [(448, 448), (1024, 1024), (1024, 768), (800, 1200)]:
    h, w = size
    # 对齐到 patch*merge = 28 的倍数（round，不是 floor）
    h2 = round(h / (PATCH * MERGE)) * (PATCH * MERGE)
    w2 = round(w / (PATCH * MERGE)) * (PATCH * MERGE)
    n = (h2 // PATCH // MERGE) * (w2 // PATCH // MERGE)
    print(f"{h}x{w}  缩放后 {h2}x{w2}  N={n:<6} 占2048序列 {n/2048:.0%}")

## 2. 官方校验（需要下载 Qwen2.5-VL 的 processor）

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.minivlm.processor", "--check"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)
print("\n→ 全部 ✓ 就说明你搞懂了；有 ✗ 就回去看 smart_resize 的取整")

## 3. 视觉 token 经济账

这一步是**连接「今天」和「第 13 天的显存工程」**的关键。

In [ ]:
print(f"{'图片尺寸':<12} {'visual token':>12} {'占2048':>8} {'单图激活(bf16,估)':>18}")
print("-" * 56)
for size in [(224,224), (448,448), (768,768), (1024,1024)]:
    h, w = size
    n = (h // 28) * (w // 28)
    # 粗略估算：每层激活 ~ n * d_model * 2 bytes（只算一层，实际要乘层数）
    act_mb = n * 2048 * 2 / 1024**2
    print(f"{h}×{w:<8} {n:>12} {n/2048:>7.0%} {act_mb:>15.1f} MB")

print("\n→ 一张 1024² 的图 = 1369 个 token，比很多人默认的「256 个」多 5 倍。")
print("  这是 VLM 训练显存爆炸的头号原因，Day 13 会算这笔账。")

## 4. M1 白板自检

拿张纸画出：**图 → 缩放 → patch → ViT → merge → 连接器 → 拼进文本序列 → LLM → 回答**。

每一环标注：张量形状 + 在哪一步 token 数从多少变成多少。

画不出来就回 Day 2/3 重看 —— 这是 W1 唯一的硬骨头。

> 今天先自己画一遍。**Day 6 会正式复盘并确认 M1** —— 那时候你要能不看任何资料讲出来。

In [ ]:
print("""今日打卡
─────────────────────────────────────────
[学到] M-RoPE 的 t/h/w 分别编码 ______；1024²图占序列 ______%
[产出] processor.py --check 全绿；白板数据流草图（拍照存 assets/）
[卡住] ______
─────────────────────────────────────────
W1 还差 Day 5（拼 Mini-VLM）+ Day 6（复盘确认 M1），明天继续""")

## 验收清单

- [ ] **`make day4` 通过** —— 我们算的 token 数等于官方 `<|image_pad|>` 数
- [ ] 能画图解释 M-RoPE 的 t/h/w 三维分别编码什么
- [ ] 能说清「1024² 图占 67% 序列」对训练意味着什么（→ batch 只能很小）
- [ ] **W1 最难的一环啃下来了**：能白板画出完整 VLM 数据流并讲清每一环（M1 的正式确认在 Day 6 的复盘日）

**卡住了？** 回看 [`days/day-04.md`](../days/day-04.md) 第五节「容易踩的坑」。

> **明天**：`days/day-05.md` —— 从零手搭 Mini-VLM：把前四天的零件拼起来